# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import duckdb
from huggingface_hub import whoami
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute(f"""
CREATE OR REPLACE SECRET hf (
TYPE huggingface,
TOKEN '{HF_TOKEN}'
)
""")

REL = "hf://datasets/FlyRank/internship-warehouse"

In [2]:
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE HUGGINGFACE, TOKEN '{HF_TOKEN}');")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients': f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content': f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':  f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
}

print("DuckDB connected and Hugging Face secret configured.")

DuckDB connected and Hugging Face secret configured.


In [3]:
query = f"""
WITH march_features AS (
    SELECT
        content_hash_id,
        client_hash_id,
        SUM(gsc_impressions) AS impressions,
        SUM(gsc_clicks) AS clicks,
        AVG(gsc_avg_position) AS avg_position,
        SUM(ga4_sessions) AS sessions,
        CASE
            WHEN SUM(gsc_impressions) = 0 THEN 0.0
            ELSE 100.0 * SUM(gsc_clicks) / SUM(gsc_impressions)
        END AS ctr
    FROM {TABLES['fact_daily']}
    WHERE month = '2026-03'
      AND gsc_data_available IS TRUE
      AND ga4_data_available IS TRUE
    GROUP BY content_hash_id, client_hash_id
),
april_performance AS (
    SELECT
        content_hash_id,
        -- Define Target Label: High search visibility but underperforming outcome in subsequent month
        CASE
            WHEN SUM(gsc_impressions) >= 1000
             AND (100.0 * SUM(gsc_clicks) / NULLIF(SUM(gsc_impressions), 0)) < 1.0
            THEN 1
            ELSE 0
        END AS target_label
    FROM {TABLES['fact_daily']}
    WHERE month = '2026-04'
      AND gsc_data_available IS TRUE
    GROUP BY content_hash_id
)
SELECT
    f.content_hash_id,
    f.client_hash_id,
    f.impressions,
    f.clicks,
    f.avg_position,
    f.sessions,
    f.ctr,
    COALESCE(p.target_label, 0) AS target
FROM march_features f
LEFT JOIN april_performance p ON f.content_hash_id = p.content_hash_id;
"""

print("Executing SQL query for March -> April temporal panel...")
df_panel = con.sql(query).df()
print(f"Dataset shape: {df_panel.shape}")
print(f"Target distribution:\n{df_panel['target'].value_counts(normalize=True)}")

Executing SQL query for March -> April temporal panel...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Dataset shape: (63856, 8)
Target distribution:
target
0    0.553668
1    0.446332
Name: proportion, dtype: float64


## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

# Finding 1: The Freshness Multiplier

**Claim:** Refreshing mature content results in a significant improvement in health score and impressions.

## Where does the label come from?

The paper groups pages according to how recently they were updated and compares refreshed pages with older, unchanged pages. However, it does not clearly explain what qualifies as a **"refresh."** For example, it is not clear whether a refresh means rewriting large sections of content, updating statistics, fixing metadata, or making only minor edits. A clearer definition of this label would make it easier for others to reproduce the analysis and apply the same methodology.

## Does the validation design carry the claim?

The comparison shows a strong relationship between refreshed pages and improved performance, but it remains observational. It is possible that the refreshed pages were already considered important or had higher traffic potential before being updated. A stronger validation design could compare the same pages before and after the refresh while controlling for factors such as seasonality, topic popularity, and existing authority. The paper appropriately limits its conclusion to patterns observed within its own dataset rather than presenting it as a universal rule, which makes the claim more balanced.

---

# Finding 2: AI-Generated Content Is Not Penalized

**Claim:** The dataset does not show evidence of a blanket penalty against AI-generated content.

## Where does the label come from?

The paper explains that most of the portfolio consists of AI-generated content created using multiple AI models. However, it does not describe in detail how content was identified or labeled as AI-generated. Knowing whether these labels came from internal publishing records, metadata, or another process would improve transparency and help others reproduce the analysis. :contentReference

## Does the validation design carry the claim?

The analysis compares different AI-generated content cohorts after accounting for content age, which supports the conclusion that there is no obvious blanket penalty within this portfolio. However, because the study mainly compares AI-generated content with other AI-generated content, there is little opportunity to compare against a human-written baseline. Adding a comparable human-authored group would strengthen the validation and allow broader conclusions. As written, the paper appropriately limits its claim to the evidence available in its own dataset rather than making a universal statement about all AI-generated content.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

## 2. My Model Under an Honest Split (Before vs After)

In Week 5, I evaluated the model using a stratified random train/validation split. Although stratification preserved the target distribution, pages from the same client could appear in both the training and validation sets. This may allow the model to learn client-specific patterns, producing optimistic evaluation metrics.

To make the evaluation more realistic, I re-ran the experiment using a grouped split based on `client_hash_id`. With this strategy, all pages from a client appear entirely in either the training set or the validation set. This better reflects deployment, where the model is expected to generalize to unseen clients rather than memorizing characteristics of clients already observed during training.

The same features, target definition, model configuration, and evaluation metric (Precision@20, Precision@50, ROC-AUC) are used so that only the validation strategy changes.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
import pandas as pd
import numpy as np

feature_cols = [
    "impressions",
    "clicks",
    "avg_position",
    "sessions",
    "ctr"
]

X = df_panel[feature_cols]
y = df_panel["target"]
groups = df_panel["client_hash_id"]

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, val_idx = next(gss.split(X, y, groups))

X_train = X.iloc[train_idx]
X_val = X.iloc[val_idx]

y_train = y.iloc[train_idx]
y_val = y.iloc[val_idx]

rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=6,
    random_state=42
)

rf.fit(X_train, y_train)

rf_probs = rf.predict_proba(X_val)[:,1]

def precision_at_k(y_true, scores, k):
    order = np.argsort(scores)[::-1][:k]
    return y_true.iloc[order].mean()

results = pd.DataFrame({
    "Metric":[
        "Precision@20",
        "Precision@50",
        "ROC-AUC"
    ],
    "Week-5 Random Split":[
        1.0000,
        1.0000,
        0.9399
    ],
    "Grouped Split":[
        precision_at_k(y_val, rf_probs,20),
        precision_at_k(y_val, rf_probs,50),
        roc_auc_score(y_val, rf_probs)
    ]
})

results

,Metric,Week-5 Random Split,Grouped Split
0,Precision@20,1.0000,1.000000
1,Precision@50,1.0000,1.000000
2,ROC-AUC,0.9399,0.902117


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

## 3. Leakage Audit

A feature leakage audit was performed before interpreting model performance.

### Features Used

- impressions
- clicks
- avg_position
- sessions
- ctr

These variables are calculated exclusively from March 2026 data and are therefore available before the April prediction period.

### Features Excluded

The following information was intentionally excluded because it contains future information or directly defines the target.

- April impressions
- April clicks
- April CTR
- April performance statistics
- Any variable computed after March 2026

The target is constructed from April performance while all predictor variables come from March, preventing direct future leakage.

Although impressions and CTR appear in both March (features) and April (target), they originate from different months. Therefore, this notebook measures temporal signal persistence rather than directly using future observations during prediction.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
audit = pd.DataFrame({

    "Feature":[
        "impressions",
        "clicks",
        "avg_position",
        "sessions",
        "ctr"
    ],

    "Available Before Prediction":[
        "Yes",
        "Yes",
        "Yes",
        "Yes",
        "Yes"
    ],

    "Leakage Risk":[
        "No",
        "No",
        "No",
        "No",
        "No"
    ]

})

audit

,Feature,Available Before Prediction,Leakage Risk
0,impressions,Yes,No
1,clicks,Yes,No
2,avg_position,Yes,No
3,sessions,Yes,No
4,ctr,Yes,No


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*


### Original Claim

"The Random Forest accurately predicts which pages need to be refreshed."

### Revised Claim

The Random Forest model ranks pages according to their estimated probability of matching the April refresh target based on March search performance signals. Under the evaluated validation design, the model achieved higher Precision@20 and Precision@50 than the hand-written baseline.

These results should be interpreted as evidence that historical search signals remain informative across consecutive months. They should not be interpreted as proof that the model predicts long-term business outcomes or guarantees future traffic improvements.

This model is intended as a decision-support tool for prioritizing pages for manual review rather than an automated decision-making system.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.